# Unstructured mesh source

This example makes use of a DAGMC unstructured tet mesh to produce a source with a MeshSpatial distribution.

In [1]:
import openmc
from pathlib import Path
from cad_to_dagmc import CadToDagmc
from openmc_source_plotter import plot_source_position
# Setting the cross section path to the correct location in the docker image.
# If you are running this outside the docker image you will have to change this path to your local cross section path.
openmc.config['cross_sections'] = Path.home() / 'nuclear_data' / 'cross_sections.xml'

# this section loads a CAD step file and creates an unstrucutred DAGMC tet mesh
# the resulting mesh file (umesh.mesh) is already included in the repo
# so this creation from step file is included for completeness but can be skipped
cad = CadToDagmc()
cad.add_stp_file('plasma_simplified_180.step')   
cad.export_unstructured_mesh_file(filename="umesh.vtk", max_mesh_size=100, min_mesh_size=10)

# Setting the cross section path to the correct location in the docker image.
# If you are running this outside the docker image you will have to change this path to your local cross section path.
openmc.config['cross_sections'] = Path.home() / 'nuclear_data' / 'cross_sections.xml'

umesh = openmc.UnstructuredMesh(filename="umesh.vtk",library='moab')

surf1 = openmc.Sphere(r=50000, boundary_type="vacuum")
region1 = -surf1

cell1 = openmc.Cell(region=region1)

my_geometry = openmc.Geometry([cell1])

my_source = openmc.IndependentSource()
my_source.angle = openmc.stats.Isotropic()
my_source.energy = openmc.stats.Discrete([14e6], [1])
# link to docs for MeshSpatial
# https://docs.openmc.org/en/latest/pythonapi/generated/openmc.stats.MeshSpatial.html
# allows us to apply the same source to each element in the mesh. The source can be varied in terms of strength
my_source.space = openmc.stats.MeshSpatial(
    mesh=umesh,
    #we set the strengths to sum to 1 to make post processing easier.
    # in a more accurate plasma source the strength could be adjusted based on the source position.
    strengths=[1/1104]*1104,
    volume_normalized=False
)

my_settings = openmc.Settings()
my_settings.batches = 10
my_settings.particles = 1000
my_settings.run_mode = "fixed source"
my_settings.source = my_source

model = openmc.model.Model(my_geometry, None, my_settings )

model.run()


# plotting the mesh source
plot = plot_source_position([my_source], n_samples=10000)
plot.show()

/home/jon/.neutronicsworkshop_3.12/lib/python3.12/site-packages/openmc_source_plotter/core.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Circle)
Info    : [  0%] Meshing curve 2 (BSpline)
Info    : [  0%] Meshing curve 3 (BSpline)
Info    : Done meshing 1D (Wall 0.0406651s, CPU 0.075045s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 3 (Plane, MeshAdapt)
Info    : [  0%] Meshing surface 2 (Plane, MeshAdapt)
Info    : [  0%] Meshing surface 1 (Surface of Revolution, MeshAdapt)


Info    : Done meshing 2D (Wall 0.873499s, CPU 0.877097s)
Info    : Meshing 3D...
Info    : 3D Meshing 1 volume with 1 connected component
Info    : Tetrahedrizing 308 nodes...
Info    : Done tetrahedrizing 316 nodes (Wall 0.00582352s, CPU 0.005226s)
Info    : Reconstructing mesh...
Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges
Info    :  - Recovering boundary
Info    : Done reconstructing mesh (Wall 0.0124025s, CPU 0.011722s)
Info    : Found volume 1
Info    : It. 0 - 0 nodes created - worst tet radius 1.62031 (nodes removed 0 0)
Info    : 3D refinement terminated (356 nodes total):
Info    :  - 0 Delaunay cavities modified for star shapeness
Info    :  - 0 nodes could not be inserted
Info    :  - 1132 tetrahedra created in 0.00666945 sec. (169729 tets/s)
Info    : 0 node relocations
Info    : Done meshing 3D (Wall 0.0356273s, CPU 0.03545s)
Info    : Optimizing mesh...
Info    : Optimizing volume 1
Info    : Optimization starts (volume = 1.78813e+08) with 


 =======================>     TIMING STATISTICS     <=======================

 Total time for initialization     = 3.2868e-02 seconds
   Reading cross sections          = 1.8175e-05 seconds
 Total time in simulation          = 2.8056e-02 seconds
   Time in transport only          = 1.1007e-02 seconds
   Time in active batches          = 2.8056e-02 seconds
   Time accumulating tallies       = 6.1380e-06 seconds
   Time writing statepoints        = 1.6443e-02 seconds
 Total time for finalization       = 3.2470e-06 seconds
 Total time elapsed                = 6.1056e-02 seconds
 Calculation Rate (active)         = 356427.0 particles/second

 ============================>     RESULTS     <============================

 Leakage Fraction            = 1.00000 +/- 0.00000

